# 归一化


In [2]:
import torch
import torch.nn as nn

def manual_layer_norm(x, gamma, beta, eps=1e-5):

    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_normalized = (x - mean) / torch.sqrt(var + eps)
    out = gamma * x_normalized + beta
    return out

# --- 测试与验证 ---
d = 5
x = torch.randn(2, d) # Batch size 2, dimension 5
# 初始化可学习参数
gamma = torch.ones(d)
beta = torch.zeros(d)
# 手动实现
y_manual = manual_layer_norm(x, gamma, beta)

# 官方 API
layer_norm = nn.LayerNorm(d)
y_official = layer_norm(x)
print("Manual LayerNorm Output:\n", y_manual)
print("Official LayerNorm Output:\n", y_official)
# 验证两者输出是否接近

Manual LayerNorm Output:
 tensor([[-0.5437,  0.4656, -1.6773,  1.1361,  0.6193],
        [-0.6039,  0.6729, -1.6889,  0.9049,  0.7150]])
Official LayerNorm Output:
 tensor([[-0.5437,  0.4656, -1.6773,  1.1361,  0.6193],
        [-0.6039,  0.6729, -1.6889,  0.9049,  0.7150]],
       grad_fn=<NativeLayerNormBackward0>)


* 中心化不变性：偏置不敏感
* 缩放不变性：缓解梯度爆炸/消失

In [4]:
import torch
import torch.nn.functional as F

# 定义输入
d = 10
x = torch.randn(d)

# 1. 验证重中心化不变性 (Re-centering Invariance)
delta = 100.0  # 一个巨大的偏移量
ln_original = F.layer_norm(x, x.shape)
ln_shifted = F.layer_norm(x + delta, x.shape)

# 比较结果
diff_center = torch.abs(ln_original - ln_shifted).max().item()
print(f"加偏移量后的最大误差: {diff_center:.2e} (应接近 0)")


# 2. 验证重缩放不变性 (Re-scaling Invariance)
lambda_val = 500.0  # 缩放因子
ln_scaled = F.layer_norm(x * lambda_val, x.shape)

# 比较结果
diff_scale = torch.abs(ln_original - ln_scaled).max().item()
print(f"乘缩放因子后的最大误差: {diff_scale:.2e} (应接近 0)")

# 输出示例:
# 加偏移量后的最大误差: 0.00e+00
# 乘缩放因子后的最大误差: 1.19e-07 (由于浮点数精度限制，非常接近0)

加偏移量后的最大误差: 6.41e-06 (应接近 0)
乘缩放因子后的最大误差: 1.01e-05 (应接近 0)


* Post-Norm 与 Pre-Norm 的博弈：相对于BLOCK的位置问题？

In [ ]:
# 在 Google 最初的论文《Attention Is All You Need》以及 BERT 模型中，采用的是 Post-Norm 结构。
# 输出之前对ADD结果进行 LayerNorm 处理，公式如下：
# y = LayerNorm(x + Sublayer(x))
# 这种结构的优点是每个子层的输出都经过规范化，有助于稳定训练过程，尤其是在深层网络中。
# 但是，Post-Norm 结构在训练非常深的模型时可能会遇到梯度消失的问题。
# 为了解决这个问题，后来提出了 Pre-Norm 结构，即在子层输入之前进行 LayerNorm 处理，公式如下：
# y = x + Sublayer(LayerNorm(x))
# 永远保留一个旁路不做规范化，能够更好地保持梯度流动，尤其是在深层网络中。

* RMSNorm：极简主义的效率革命


* 驯服注意力机制：QK-Norm 与熵坍塌：训练稳定，减少尖峰（直接变成独热编码了）

## 专业化变体
* Sandwich-Norm：两个Norm夹着ATTENTION或者MLP
* NormFormer：通过“过归一化”加速收敛

In [ ]:
class RMSNorm(torch.nn.Module):
    """
    RMSNorm (Root Mean Square Layer Normalization)
    
    RMSNorm 是 LayerNorm 的简化版本，只对输入进行缩放，不进行中心化（不减去均值）。
    相比 LayerNorm，RMSNorm 计算更高效，且在实际应用中效果相当。
    
    公式：
        RMSNorm(x) = weight * (x / sqrt(mean(x^2) + eps))
    
    与 LayerNorm 的区别：
        - LayerNorm: (x - mean(x)) / sqrt(var(x) + eps)
        - RMSNorm: x / sqrt(mean(x^2) + eps)
        - RMSNorm 不减去均值，只进行缩放，计算更快
    """
    def __init__(self, dim: int, eps: float = 1e-5):
        """
        初始化 RMSNorm 层
        
        Args:
            dim: 输入特征的维度
            eps: 防止除零的小常数（默认 1e-5）
        """
        super().__init__()
        self.eps = eps
        # 可学习的缩放参数，初始化为全 1
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        """
        计算 RMS 归一化
        
        公式：x / sqrt(mean(x^2) + eps)
        
        Args:
            x: 输入张量 [..., dim]
            
        Returns:
            归一化后的张量，形状与输入相同
        """
        # x.pow(2): 计算 x 的平方
        # .mean(-1, keepdim=True): 在最后一个维度上求均值，保持维度
        # torch.rsqrt: 计算 1/sqrt，比先 sqrt 再除更快
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        """
        前向传播
        
        Args:
            x: 输入张量，可以是任意精度（float16, bfloat16, float32）
            
        Returns:
            归一化并缩放后的张量，保持原始精度
        """
        # 先转换为 float32 进行归一化计算（提高数值稳定性）
        # 然后转换回原始精度（type_as(x)）
        # 最后乘以可学习的权重
        return self.weight * self._norm(x.float()).type_as(x)